# Segmentation alimentaire avec FoodSeg103 et YOLOv11-seg

## Objectif

Ce notebook entraîne un modèle de segmentation alimentaire à partir du dataset FoodSeg103.

L’objectif est de segmenter les zones alimentaires dans une image afin d’estimer la proportion occupée par les aliments dans un plat.

## Dataset

Source : FoodSeg103 sur Kaggle  
Chemin Kaggle : `/kaggle/input/datasets/fontainenathan/foodseg103`

Le dataset contient :
- des images dans `Images/img_dir`
- des masques de segmentation dans `Images/ann_dir`
- les listes d’entraînement et de test dans `ImageSets`

## Sorties

Le notebook génère :
- un dataset converti au format YOLO segmentation
- un modèle entraîné `best.pt`
- des résultats de validation

In [1]:
# Installation de la bibliothèque Ultralytics pour utiliser YOLO.
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


In [2]:
# Import des librairies nécessaires.
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from IPython.display import FileLink

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Chemin racine du dataset FoodSeg103.
ROOT = "/kaggle/input/datasets/fontainenathan/foodseg103"

# Dossiers des images et des masques.
IMG_DIR = f"{ROOT}/Images/img_dir"
ANN_DIR = f"{ROOT}/Images/ann_dir"

# Fichier des catégories.
CATEGORY_FILE = f"{ROOT}/category_id.txt"

# Listes train/test.
TRAIN_LIST = f"{ROOT}/ImageSets/train.txt"
TEST_LIST = f"{ROOT}/ImageSets/test.txt"

# Dossier de sortie converti au format YOLO.
DEST = "/kaggle/working/dataset"

print("ROOT:", ROOT)
print("IMG_DIR:", IMG_DIR)
print("ANN_DIR:", ANN_DIR)
print("CATEGORY_FILE:", CATEGORY_FILE)
print("TRAIN_LIST:", TRAIN_LIST)
print("TEST_LIST:", TEST_LIST)


ROOT: /kaggle/input/datasets/fontainenathan/foodseg103
IMG_DIR: /kaggle/input/datasets/fontainenathan/foodseg103/Images/img_dir
ANN_DIR: /kaggle/input/datasets/fontainenathan/foodseg103/Images/ann_dir
TRAIN_LIST: /kaggle/input/datasets/fontainenathan/foodseg103/ImageSets/train.txt
TEST_LIST: /kaggle/input/datasets/fontainenathan/foodseg103/ImageSets/test.txt


In [4]:
# Lecture des fichiers train.txt et test.txt.
# Les noms peuvent contenir déjà l'extension .jpg.
def lire_liste(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_files = lire_liste(TRAIN_LIST)
val_files = lire_liste(TEST_LIST)

print("Nombre train:", len(train_files))
print("Nombre val:", len(val_files))
print("Exemples train:", train_files[:5])
print("Exemples val:", val_files[:5])

Nombre train: 4983
Nombre val: 2135
Exemples train: ['00000000.jpg', '00000001.jpg', '00000002.jpg', '00000003.jpg', '00000004.jpg']
Exemples val: ['00000048.jpg', '00000263.jpg', '00001977.jpg', '00002106.jpg', '00004401.jpg']


In [ ]:
# Lecture des catégories FoodSeg103.
# Le masque utilise : 0 = background, 1 à 103 = classes alimentaires.
# YOLO doit utiliser : 0 à 102 = classes alimentaires, sans background.

categories_foodseg = {}

with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)  # sépare l'ID et le nom, même si le nom contient des espaces
        if len(parts) == 2:
            id_foodseg = int(parts[0])
            nom_classe = parts[1].strip()
            categories_foodseg[id_foodseg] = nom_classe

# On ignore le background et on décale les IDs pour YOLO.
categories_yolo = {
    id_foodseg - 1: nom
    for id_foodseg, nom in categories_foodseg.items()
    if id_foodseg != 0
}

print("Nombre de classes YOLO:", len(categories_yolo))
print("Exemples:")
for i in list(categories_yolo.keys())[:10]:
    print(i, "->", categories_yolo[i])

# Exemple important : FoodSeg 66 = rice, donc YOLO 65 = rice.
print("FoodSeg 48 -> YOLO", 48 - 1, "=", categories_foodseg.get(48))
print("FoodSeg 66 -> YOLO", 66 - 1, "=", categories_foodseg.get(66))
print("FoodSeg 90 -> YOLO", 90 - 1, "=", categories_foodseg.get(90))


In [5]:
# Création de la structure attendue par YOLO.
for d in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{DEST}/{d}", exist_ok=True)

print("Structure YOLO créée.")

Structure YOLO créée.


In [6]:
# Conversion d'un masque PNG en fichier label YOLO segmentation multi-classes.
# Important : on conserve les vraies catégories du masque FoodSeg103.
# 0 = background, ignoré.
# 1 à 103 = classes alimentaires FoodSeg103.
# YOLO attend des classes de 0 à 102, donc : classe_yolo = valeur_masque - 1.

def mask_to_yolo_multiclass(mask_path, output_txt, min_area=20):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return False

    h, w = mask.shape
    lignes = []

    valeurs_classes = np.unique(mask)

    for valeur in valeurs_classes:
        if valeur == 0:
            continue  # background

        classe_yolo = int(valeur) - 1

        # Sécurité : ignorer une classe inconnue.
        if classe_yolo not in categories_yolo:
            continue

        masque_classe = (mask == valeur).astype(np.uint8) * 255

        contours, _ = cv2.findContours(
            masque_classe,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < 3:
                continue

            if cv2.contourArea(contour) < min_area:
                continue

            # Simplification légère pour éviter des polygones trop longs.
            epsilon = 0.001 * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, epsilon, True)

            if len(contour) < 3:
                continue

            points = contour.reshape(-1, 2)
            coords = []

            for x, y in points:
                coords.append(x / w)
                coords.append(y / h)

            ligne = str(classe_yolo) + " " + " ".join([f"{c:.6f}" for c in coords])
            lignes.append(ligne)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lignes))

    return len(lignes) > 0


In [7]:
# Fonction pour trouver le masque correspondant à une image.
# Exemple : 00000001.jpg -> 00000001.png

def trouver_masque(nom_image, split):
    base = Path(nom_image).stem
    mask_path = f"{ANN_DIR}/{split}/{base}.png"

    if os.path.exists(mask_path):
        return mask_path

    return None


# Conversion d'un split train ou test vers la structure YOLO.
def convertir_split(files, src_split, dst_split):
    ok = 0
    manquants = 0
    sans_masque_valide = 0

    for nom_image in files:
        img_path = f"{IMG_DIR}/{src_split}/{nom_image}"
        mask_path = trouver_masque(nom_image, src_split)

        if not os.path.exists(img_path) or mask_path is None:
            manquants += 1
            continue

        base = Path(nom_image).stem

        dst_img = f"{DEST}/images/{dst_split}/{nom_image}"
        dst_label = f"{DEST}/labels/{dst_split}/{base}.txt"

        shutil.copy(img_path, dst_img)

        if mask_to_yolo_multiclass(mask_path, dst_label):
            ok += 1
        else:
            sans_masque_valide += 1

    print(f"{dst_split} ok:", ok)
    print(f"{dst_split} manquants:", manquants)
    print(f"{dst_split} sans masque valide:", sans_masque_valide)


convertir_split(train_files, "train", "train")
convertir_split(val_files, "test", "val")


train ok: 4983
train manquants: 0
train sans masque valide: 0
val ok: 2135
val manquants: 0
val sans masque valide: 0


In [8]:
# Vérification finale de la conversion.
print("Images train:", len(os.listdir(f"{DEST}/images/train")))
print("Labels train:", len(os.listdir(f"{DEST}/labels/train")))
print("Images val:", len(os.listdir(f"{DEST}/images/val")))
print("Labels val:", len(os.listdir(f"{DEST}/labels/val")))

print("Exemples labels:", os.listdir(f"{DEST}/labels/train")[:5])

Images train: 4983
Labels train: 4983
Images val: 2135
Labels val: 2135
Exemples labels: ['00002003.txt', '00003163.txt', '00000847.txt', '00004331.txt', '00006775.txt']


In [9]:
# Vérification du contenu d'un fichier label.
exemple_label = os.listdir(f"{DEST}/labels/train")[0]

with open(f"{DEST}/labels/train/{exemple_label}", "r") as f:
    print(f.read()[:500])

0 0.140625 0.000000 0.140625 0.002967 0.138672 0.005935 0.138672 0.011869 0.136719 0.014837 0.136719 0.017804 0.134766 0.020772 0.134766 0.023739 0.130859 0.029674 0.130859 0.032641 0.126953 0.038576 0.126953 0.041543 0.125000 0.044510 0.125000 0.050445 0.123047 0.053412 0.123047 0.068249 0.121094 0.071217 0.121094 0.077151 0.119141 0.080119 0.117188 0.077151 0.111328 0.077151 0.109375 0.074184 0.103516 0.074184 0.101562 0.071217 0.095703 0.071217 0.093750 0.068249 0.085938 0.068249 0.083984 0.0


In [10]:
# Création du fichier de configuration YOLO multi-classes.
# On utilise les 103 classes alimentaires de FoodSeg103, sans le background.

yaml_content = f"""
path: {DEST}
train: images/train
val: images/val

names:
"""

for idx in sorted(categories_yolo.keys()):
    nom = categories_yolo[idx].replace("'", "")
    yaml_content += f"  {idx}: '{nom}'\n"

with open("/kaggle/working/dataset.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(yaml_content[:1500])



path: /kaggle/working/dataset
train: images/train
val: images/val

names:
  0: food



In [11]:
# Chargement du modèle YOLO segmentation pré-entraîné.
# Pour tester plus vite, tu peux garder yolo11n-seg.pt.
# Pour un meilleur résultat final, tu peux utiliser yolo11m-seg.pt.

model = YOLO("yolo11n-seg.pt")

# Entraînement du modèle multi-classes.
model.train(
    data="/kaggle/working/dataset.yaml",
    task="segment",
    epochs=50,
    imgsz=640,
    batch=8
)


Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7940e8c634d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [12]:
# Évaluation du modèle sur les données de validation.
metrics = model.val()
print(metrics)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 752.0±534.2 MB/s, size: 52.8 KB)
val: Scanning /kaggle/working/dataset/labels/val.cache... 2135 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2135/2135 110.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 6.2it/s 21.5s
                   all       2135       3234      0.925      0.882      0.935      0.852      0.926      0.884      0.935      0.811
Speed: 0.8ms preprocess, 4.2ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /kaggle/working/runs/segment/val
ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytic

In [13]:
# Test sur des images de validation.
model.predict(
    source=f"{DEST}/images/val",
    conf=0.25,
    save=True,
    max_det=10
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2135 /kaggle/working/dataset/images/val/00000048.jpg: 608x640 1 food, 69.2ms
image 2/2135 /kaggle/working/dataset/images/val/00000263.jpg: 608x640 1 food, 10.1ms
image 3/2135 /kaggle/working/dataset/images/val/00001977.jpg: 480x640 5 foods, 66.6ms
image 4/2135 /kaggle/working/dataset/images/val/00002106.jpg: 480x640 1 food, 11.6ms
image 5/2135 /kaggle/working/dataset/images/val/00004401.jpg: 640x640 3 foods, 13.7ms
image 6/2135 /kaggle/working/da

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'food'}
 obb: None
 orig_img: array([[[131, 147, 183],
         [126, 144, 181],
         [121, 140, 178],
         ...,
         [ 44,  91, 177],
         [ 47,  95, 183],
         [ 48,  98, 186]],
 
        [[128, 146, 183],
         [124, 143, 181],
         [120, 140, 181],
         ...,
         [ 33,  78, 165],
         [ 43,  89, 177],
         [ 49,  97, 185]],
 
        [[123, 143, 184],
         [121, 140, 183],
         [118, 139, 184],
         ...,
         [ 34,  75, 160],
         [ 49,  92, 179],
         [ 62, 105, 192]],
 
        ...,
 
        [[202, 187, 195],
         [201, 188, 196],
         [213, 205, 212],
         ...,
         [231, 220, 206],
         [231, 220, 206],
         [231, 220, 206]],
 
        [[198, 183, 191],
         [196, 183, 191],
         [203, 195, 202],

In [14]:
# Trouver le modèle entraîné.
!find /kaggle/working -name "best.pt"

/kaggle/working/runs/segment/train/weights/best.pt


In [15]:
# Copier best.pt dans un emplacement simple.
!cp /kaggle/working/runs/segment/train*/weights/best.pt /kaggle/working/best.pt

In [ ]:
# Créer un lien de téléchargement.
FileLink("/kaggle/working/best.pt")

/kaggle/working/best.pt